In [1]:
import pickle

In [2]:
with open('/root/autodl-tmp/chuandian_eq/data/taxi/raw/train.pkl', 'rb') as f:
    data = pickle.load(f)

In [3]:
print(data['train'][0])
print(len(data['train']))
print(data['dim_process'])

[{'idx_event': 1, 'type_event': 8, 'time_since_start': 0.0, 'time_since_last_event': 0.0}, {'idx_event': 2, 'type_event': 3, 'time_since_start': 0.07888888888888888, 'time_since_last_event': 0.07888888888888888}, {'idx_event': 3, 'type_event': 8, 'time_since_start': 0.27666666666666667, 'time_since_last_event': 0.19777777777777777}, {'idx_event': 4, 'type_event': 3, 'time_since_start': 0.37972222222222224, 'time_since_last_event': 0.10305555555555557}, {'idx_event': 5, 'type_event': 8, 'time_since_start': 0.5475, 'time_since_last_event': 0.16777777777777775}, {'idx_event': 6, 'type_event': 3, 'time_since_start': 1.0013888888888889, 'time_since_last_event': 0.4538888888888889}, {'idx_event': 7, 'type_event': 8, 'time_since_start': 1.4066666666666667, 'time_since_last_event': 0.40527777777777785}, {'idx_event': 8, 'type_event': 3, 'time_since_start': 1.8002777777777779, 'time_since_last_event': 0.39361111111111113}, {'idx_event': 9, 'type_event': 8, 'time_since_start': 1.8716666666666666

In [4]:
from src.data.sequence import Sequence
import torch
def list_of_dicts_to_sequence(event_list):
    inter_times = [event['time_since_last_event'] for event in event_list]
    inter_times = torch.tensor(inter_times, dtype=torch.float32)
    t_start = 0.0
    t_nll_start = 0.0
    arrival_times = [event['time_since_start'] for event in event_list]
    type_event = [event['type_event'] for event in event_list]
    type_event = torch.tensor(type_event, dtype=torch.long)
    return Sequence(
        t_start=t_start,
        t_nll_start=t_nll_start,
        arrival_times=arrival_times,
        inter_times=inter_times,
        type_event=type_event
    )


In [5]:
sequence_list = [list_of_dicts_to_sequence(s) for s in data["train"]]

/root/autodl-tmp/chuandian_eq/src/data/sequence.py:182: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


In [6]:
from src.data.batch import Batch

In [7]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset(sequence_list)
loader = ds.get_dataloader(
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [8]:
for batch in loader:
    print(batch.keys())
    break

['inter_times', 'arrival_times', 't_start', 't_end', 't_nll_start', 'nll_mask', 'start_idx', 'end_idx', 'non_pad_mask', 'type_seq', 'type_event']


In [9]:
batch.type_event[6]

tensor([8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 1,
        8, 3, 5, 0, 5, 3, 8, 0, 8, 3, 8, 3, 8, 1])

In [10]:
batch.type_seq

tensor([[   5,    3,    8,  ...,    1, -100, -100],
        [   8,    3,    8,  ...,    1,    8,    3],
        [   8,    3,    8,  ...,    3,    5,    1],
        ...,
        [   8,    0,    5,  ...,    3, -100, -100],
        [   5,    3,    8,  ...,    0, -100, -100],
        [   8,    3,    8,  ...,    3,    8,    3]])

In [11]:
batch.arrival_times

tensor([[ 0.0000,  0.6375,  0.7658,  ...,  7.0125,  7.0125,  7.0125],
        [ 0.0000,  0.0933,  0.2897,  ...,  6.8589,  7.1836,  7.3711],
        [ 0.0000,  0.1897,  0.2586,  ...,  9.5119, 12.2681, 12.7225],
        ...,
        [ 0.0000,  0.7172,  1.4031,  ...,  6.9553,  6.9553,  6.9553],
        [ 0.0000,  0.3175,  0.8483,  ...,  6.5350,  6.5350,  6.5350],
        [ 0.0000,  0.0628,  0.1572,  ...,  6.8953,  6.9383,  7.0403]])

In [12]:
batch.inter_times

tensor([[0.0000, 0.6375, 0.1283,  ..., 0.3625, 0.0000, 0.0000],
        [0.0000, 0.0933, 0.1964,  ..., 0.4864, 0.3247, 0.1875],
        [0.0000, 0.1897, 0.0689,  ..., 0.4894, 2.7561, 0.4544],
        ...,
        [0.0000, 0.7172, 0.6858,  ..., 0.2206, 0.0000, 0.0000],
        [0.0000, 0.3175, 0.5308,  ..., 0.1406, 0.0000, 0.0000],
        [0.0000, 0.0628, 0.0944,  ..., 0.1350, 0.0431, 0.1019]])

In [13]:
from config.config_loader import load_args_from_yaml 
args = load_args_from_yaml("config/THP.yaml")
base_dir = f"data/{args.dataset}"

In [14]:
from src.data.preparation import prepare_data_tpp
df, train_loader, val_loader, test_loader,dataset = prepare_data_tpp(
    args,
    base_dir
)

instantiating registered subclass ChuanDian-SlidingWindow of <class 'src.data.catalog.Catalog'>
Using catalog dataset class: <class 'src.catalogs.chuandian.ChuanDianSlidingWindow'>
Loading existing catalog from /root/autodl-tmp/chuandian_eq/data/ChuanDian/raw.


/root/autodl-tmp/chuandian_eq/src/data/sequence.py:182: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


Generated 1858 sliding window sequences.


In [15]:
for batch in train_loader:
    print(f"arival_times: {batch.arrival_times}")
    print(f"inter_times: {batch.inter_times}")
    print(f"type_event: {batch.type_seq}")
    print(f"non_pad_mask: {batch.non_pad_mask}")
    print(f"t_start: {batch.t_start}")
    print(f"t_end: {batch.t_end}")  
    break

arival_times: tensor([[0.0000e+00, 9.2593e-03, 1.6898e-02,  ..., 1.8970e+02, 1.9401e+02,
         1.9507e+02],
        [0.0000e+00, 1.7286e-01, 5.8794e-01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 4.7304e-01, 8.1395e-01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [0.0000e+00, 3.3622e+00, 1.1348e+01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 7.9860e+00, 9.4183e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 7.5507e+00, 1.1069e+01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00]])
inter_times: tensor([[0.0000e+00, 9.2593e-03, 7.6389e-03,  ..., 4.6502e+00, 4.3067e+00,
         1.0541e+00],
        [0.0000e+00, 1.7286e-01, 4.1508e-01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 4.7304e-01, 3.4090e-01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [0.0000e+00, 3.3622e+00, 7.9860e+00,  ..., 0.0000e+00, 0.0000e+00

In [18]:
batch[:,:-3]

EventBatch(
  arrival_times: [32, 491],
  inter_times: [32, 491],
  non_pad_mask: [32, 491],
  type_seq: [32, 491],
  t_start: [32],
  t_end: [32],
  mag: [32, 491],
  loc: [32, 491, 2],
  depth: [32, 491]
)

In [20]:
for batch in train_loader:
   print(
    batch.inter_times.max().item(),
    batch.inter_times.min().item(),
    batch.inter_times.mean().item()
)

   

20.92274284362793 0.0 0.3895527720451355
25.76251220703125 0.0 1.0898199081420898
25.76251220703125 0.0 1.8248645067214966
14.563750267028809 0.0 1.2474257946014404
17.008506774902344 0.0 0.9627341628074646
17.008506774902344 0.0 1.0691419839859009
16.067222595214844 0.0 0.6277449727058411
14.994109153747559 0.0 0.3799874484539032
8.830925941467285 0.0 1.389624834060669
10.653969764709473 0.0 1.1944912672042847
17.002674102783203 0.0 1.6923924684524536
13.115948677062988 0.0 1.8295743465423584
17.533124923706055 0.0 1.6421880722045898
20.633773803710938 0.0 2.326740264892578
19.78827476501465 0.0 2.1274466514587402
27.963159561157227 0.0 1.9628922939300537
21.427291870117188 0.0 1.8730108737945557
15.465694427490234 0.0 1.8197883367538452
17.39645767211914 0.0 2.725956916809082
25.182165145874023 0.0 2.9112558364868164
17.12474250793457 0.0 2.136190891265869
17.12474250793457 0.0 2.4270033836364746
32.8729133605957 0.0 2.260443687438965
18.65089988708496 0.0 2.6265690326690674
21.67937